# SOLID Tester — Robust Dataset Cleaner + Error Diagnosis

## 1 · Pick your principle

In [3]:
# ── Choose ONE: 'SRP' | 'OCP' | 'LSP' | 'ISP' | 'DIP' ───────────────────────
PRINCIPLE = 'SRP'

## 2 · Import the selected tool

In [4]:
import ast
import re
import traceback
import importlib
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath('..'))

TOOL_MAP = {
    'SRP': ('SOLID.SRP_Detection_Final',           'get_srp_report'),
    'OCP': ('SOLID.OCP_Detection_Final',           'get_ocp_report'),
    'LSP': ('SOLID.Liskov_Substitution_Principle', 'get_lsp_report'),
    'ISP': ('SOLID.ISP_detect',                    'get_isp_report'),
    'DIP': ('SOLID.dip_analyzer',                  'get_dip_report'),
}

assert PRINCIPLE in TOOL_MAP, f"Unknown principle '{PRINCIPLE}'. Choose from: {list(TOOL_MAP)}"
module_path, fn_name = TOOL_MAP[PRINCIPLE]

try:
    mod     = importlib.import_module(module_path)
    tool_fn = getattr(mod, fn_name)
    print(f'✅ Loaded  {fn_name}  from  {module_path}')
except (ImportError, AttributeError) as e:
    print(f'⚠️  Could not load tool: {e}')
    tool_fn = lambda code: {'status': 'Pass'}

✅ Loaded  get_srp_report  from  SOLID.SRP_Detection_Final


## 3 · Load & clean dataset

Problems found in the dataset and how each is fixed:

| Problem | Example in Excel | Fixed to |
|---|---|---|
| Escaped single quotes | `\'` | `'` |
| Escaped double quotes | `\"` | `"` |
| Escaped newlines | `\\n` | real `\n` |
| Escaped tabs | `\\t` | real `\t` |
| Escaped backslash | `\\\\` | `\\` |
| Escaped forward slash | `\/` | `/` |
| Surrounding quotes | `'class Foo:...'` | stripped |
| Quotes inside f-strings | `f\'{x}\'` | `f"{x}"` |
| Empty function bodies | `def f():\n` (no body) | adds `pass` |

In [5]:
DATA_PATH = r'python-codes.xlsx'

df = pd.read_excel(DATA_PATH, dtype=str)
df.columns = df.columns.str.strip().str.lower()


def fix_fstring_quotes(code: str) -> str:
    """
    Find f-strings that use single quotes but contain escaped single quotes
    inside them (e.g. f'{key} : {value}\\'), and rewrite them to use
    double quotes instead so the content is not mangled.

    Strategy: tokenise the code line by line. Any line that contains
    f'...' where the content has unescaped backslash+quote patterns
    gets the outer quotes swapped to double quotes.
    """
    # Match f-strings with single-quote delimiters that have \' inside
    # Pattern: f'...\' ... ' — we swap the outer quotes to double quotes
    pattern = re.compile(r"f'((?:[^'\\]|\\.)*)'")  

    def replacer(m):
        inner = m.group(1)
        # Unescape \'  inside since we're switching to double-quote delimiter
        inner = inner.replace("\\'" , "'")
        # Escape any bare double quotes that are now inside the new delimiter
        inner = inner.replace('"', '\\"')
        return f'f"{inner}"'

    return pattern.sub(replacer, code)


def add_missing_pass(code: str) -> str:
    """
    Fix 'expected an indented block after function/class/if definition'
    by inserting a `pass` after any def/class/if/else/elif/for/while/with/try
    line whose body is empty (next non-empty line is at same or lower indent).
    """
    lines  = code.splitlines()
    result = []
    block_headers = re.compile(
        r'^(\s*)(def |class |if |elif |else:|for |while |with |try:|except|finally:)'
    )

    for i, line in enumerate(lines):
        result.append(line)
        m = block_headers.match(line)
        if m and line.rstrip().endswith(':'):
            indent = len(m.group(1))
            # Look ahead for the next non-empty line
            next_content = None
            for j in range(i + 1, len(lines)):
                if lines[j].strip():
                    next_content = lines[j]
                    break
            # If no next line or next line is at same/lower indent → body missing
            if next_content is None or len(next_content) - len(next_content.lstrip()) <= indent:
                result.append(' ' * (indent + 4) + 'pass')

    return '\n'.join(result)


def clean_code(raw: str) -> str:
    if not isinstance(raw, str):
        return ''

    s = raw.strip()

    # 1. Strip surrounding quotes
    if (s.startswith("'") and s.endswith("'")) or \
       (s.startswith('"') and s.endswith('"')):
        s = s[1:-1]

    # 2. Preserve real double-backslashes before touching anything else
    s = s.replace('\\\\', '\x00BSLASH\x00')

    # 3. Unescape newlines and tabs
    s = s.replace('\\n', '\n')
    s = s.replace('\\t', '\t')

    # 4. Fix f-strings with single-quote delimiters containing \' — do this
    #    BEFORE unescaping quotes globally so the pattern is still detectable
    s = fix_fstring_quotes(s)

    # 5. Unescape remaining stray escaped quotes (outside f-strings)
    s = s.replace("\\'", "'")
    s = s.replace('\\"', '"')

    # 6. Fix escaped forward slash (invalid escape sequence warning)
    s = s.replace('\\/', '/')

    # 7. Restore real backslashes
    s = s.replace('\x00BSLASH\x00', '\\')

    # 8. Fix empty function/class bodies
    s = add_missing_pass(s)

    return s


df['code_raw'] = df['code']
df['code']     = df['code'].apply(clean_code)

print(f'Loaded {len(df)} rows  |  columns: {list(df.columns)}')
print('\n── Sample cleaned code (row 0) ──────────────────────')
print(df['code'].iloc[0])

Loaded 641 rows  |  columns: ['id', 'language', 'code', 'srp', 'ocp', 'lsp', 'isp', 'dip', 'code_raw']

── Sample cleaned code (row 0) ──────────────────────
class Writer:
    def __init__(self, type: int) -> None:
        self.type = type

    def write(self, contents: bytearray):
        if (self.type == 0):
            with open("random_file.txt", "w") as output_file:
                output_file.write(contents)
        elif (self.type == 1):
            self.some_socket.write(contents)

        elif (self.type == 2 ):
            self.db.write()

file_writer = Writer(type=0)
network_writer = Writer(type=1)


## 4 · Validate cleaning — ast.parse every row

In [6]:
parse_results = []

for _, row in df.iterrows():
    try:
        ast.parse(row['code'])
        parse_results.append({'id': row['id'], 'parseable': True,  'error': ''})
    except SyntaxError as e:
        parse_results.append({'id': row['id'], 'parseable': False, 'error': str(e)})

parse_df = pd.DataFrame(parse_results)
ok  = parse_df['parseable'].sum()
bad = (~parse_df['parseable']).sum()

print(f'✅ Parseable after cleaning : {ok}')
print(f'❌ Still broken (SyntaxError): {bad}')

if bad:
    print('\n── Rows that still fail ast.parse ───────────────────')
    still_bad = parse_df[~parse_df['parseable']].reset_index(drop=True)
    display(still_bad)

    # Show first 3 broken code snippets to diagnose remaining patterns
    bad_ids = still_bad['id'].tolist()[:3]
    for bid in bad_ids:
        snippet = df[df['id'] == bid]['code'].values[0]
        print(f'\n── id={bid} ──────────────────────────────────────────')
        print(snippet[:500])

✅ Parseable after cleaning : 634
❌ Still broken (SyntaxError): 7

── Rows that still fail ast.parse ───────────────────


<unknown>:64: SyntaxWarning: invalid escape sequence '\/'
<unknown>:61: SyntaxWarning: invalid escape sequence '\/'


,id,parseable,error
0,202,False,unterminated string literal (detected at line ...
1,289,False,unterminated string literal (detected at line ...
2,427,False,expected an indented block after function defi...
3,509,False,unterminated string literal (detected at line ...
4,566,False,"unmatched ']' (<unknown>, line 61)"
5,591,False,"invalid syntax (<unknown>, line 31)"
6,596,False,unterminated string literal (detected at line ...



── id=202 ──────────────────────────────────────────
class Animal:
    def __init__(self, animal):
        self._animal = animal

    @property
    def animal(self):
        return self._animal

    @animal.setter
    def animal(self, animal):
        self._animal = animal

    def make_sound(self):
        if self._animal == 'cat':
            return 'meow'
        elif self._animal == 'dog':
            return 'woof"


if __name__ == "__main__':
    animals = [Animal('cat'), Animal('dog')]
    for animal in animals:
        print(animal.animal, 

── id=289 ──────────────────────────────────────────
class Invoice:
  def __init__(self, customerName, total):
    self.customerName = customerName
    self.total = total
  
  def get_details(self):
    return f""'Customer name: {self.customerName}, 
Total: {self.total}'''

class SalesTax:
  def __init__(self, state):
    self.state = state

  def get_sales_tax(self):
      if self.state == 'AZ':
        return 5.5
      elif self.state == 

## 5 · Run tests

In [7]:
def extract_status(report) -> str:
    if isinstance(report, list):
        statuses = [r.get('status', 'Pass') for r in report]
        return 'Violation' if any(s == 'Violation' for s in statuses) else (statuses[0] if statuses else 'Pass')
    if isinstance(report, dict):
        return report.get('status', 'Pass')
    return 'Pass'

def label_to_expected(label: str) -> str:
    label = str(label).strip().lower()
    if label in ('non-compliant', 'violation', 'fail'): return 'Violation'
    if label in ('compliant', 'pass', 'ok'):            return 'Pass'
    return label.capitalize()


records = []

for _, row in df.iterrows():
    code     = row.get('code', '')
    row_id   = row.get('id', '?')
    expected = label_to_expected(row.get(PRINCIPLE.lower(), 'compliant'))
    exc_type = exc_msg = exc_tb = ''

    try:
        report  = tool_fn(code)
        got     = extract_status(report)
        outcome = 'ERROR' if got == 'Error' else ('PASS' if got == expected else 'FAIL')
    except Exception as exc:
        got      = 'Error'
        outcome  = 'ERROR'
        exc_type = type(exc).__name__
        exc_msg  = str(exc)
        exc_tb   = traceback.format_exc()
        report   = exc_msg

    records.append({
        'id': row_id, 'code': code, 'expected': expected,
        'got': got, 'outcome': outcome,
        'exc_type': exc_type, 'exc_msg': exc_msg, 'exc_tb': exc_tb,
        'detail': str(report) if outcome != 'PASS' else '',
    })

results_df = pd.DataFrame(records)

total   = len(results_df)
correct = (results_df['outcome'] == 'PASS').sum()
fails   = (results_df['outcome'] == 'FAIL').sum()
errors  = (results_df['outcome'] == 'ERROR').sum()

print(f'Principle : {PRINCIPLE}')
print(f'Accuracy  : {correct/total*100:.1f}%  ({correct}/{total})')
print(f'Failures  : {fails}   |   Errors: {errors}')

<unknown>:64: SyntaxWarning: invalid escape sequence '\/'
<unknown>:61: SyntaxWarning: invalid escape sequence '\/'


Principle : SRP
Accuracy  : 36.3%  (233/641)
Failures  : 401   |   Errors: 7


In [ ]:
import pandas as pd
import traceback
from sklearn.metrics import classification_report

# --------------------------------------------------
# Helpers
# --------------------------------------------------

def extract_status(report) -> str:
    if isinstance(report, list):
        statuses = [r.get('status', 'Pass') for r in report]

        return (
            'Violation'
            if any(s == 'Violation' for s in statuses)
            else (statuses[0] if statuses else 'Pass')
        )

    if isinstance(report, dict):
        return report.get('status', 'Pass')

    return 'Pass'


def label_to_expected(label: str) -> str:
    label = str(label).strip().lower()

    if label in ('non-compliant', 'violation', 'fail'):
        return 'Violation'

    if label in ('compliant', 'pass', 'ok'):
        return 'Pass'

    return label.capitalize()


# --------------------------------------------------
# Main Evaluation + Threshold Sweep
# --------------------------------------------------

def threshold_sweep(df, thresholds, tool_fn, PRINCIPLE):

    for threshold in thresholds:

        print(f"\n{'='*70}")
        print(f"PRINCIPLE: {PRINCIPLE} | THRESHOLD = {threshold:.2f}")
        print(f"{'='*70}")

        records = []

        tp = fp = tn = fn = 0

        # ------------------------------------------
        # Evaluate all rows
        # ------------------------------------------

        for _, row in df.iterrows():

            code     = row.get('code', '')
            row_id   = row.get('id', '?')

            expected = label_to_expected(
                row.get(PRINCIPLE.lower(), 'compliant')
            )

            exc_type = ''
            exc_msg  = ''
            exc_tb   = ''

            try:
                # Run detector
                report = tool_fn(code)

                # ----------------------------------
                # Extract score
                # ----------------------------------

                score = 0

                if isinstance(report, list) and len(report) > 0:
                    score = report[0].get('score', 0) / 100

                elif isinstance(report, dict):
                    score = report.get('score', 0) / 100

                # ----------------------------------
                # Threshold prediction
                # ----------------------------------

                got = (
                    'Violation'
                    if score > threshold
                    else 'Pass'
                )

                outcome = (
                    'PASS'
                    if got == expected
                    else 'FAIL'
                )

            except Exception as exc:

                got      = 'Error'
                outcome  = 'ERROR'

                exc_type = type(exc).__name__
                exc_msg  = str(exc)
                exc_tb   = traceback.format_exc()

                report   = exc_msg
                score    = 0

            # --------------------------------------
            # Update confusion counts
            # --------------------------------------

            if got != 'Error':

                if got == 'Violation' and expected == 'Violation':
                    tp += 1

                elif got == 'Pass' and expected == 'Pass':
                    tn += 1

                elif got == 'Violation' and expected == 'Pass':
                    fp += 1

                elif got == 'Pass' and expected == 'Violation':
                    fn += 1

            # --------------------------------------
            # Save record
            # --------------------------------------

            records.append({
                'id': row_id,
                'code': code,
                'expected': expected,
                'got': got,
                'score': score,
                'outcome': outcome,
                'exc_type': exc_type,
                'exc_msg': exc_msg,
                'exc_tb': exc_tb,
                'detail': str(report) if outcome != 'PASS' else '',
            })

        # ------------------------------------------
        # Create Results DataFrame
        # ------------------------------------------

        results_df = pd.DataFrame(records)

        # ------------------------------------------
        # Metrics
        # ------------------------------------------

        total   = len(results_df)
        correct = (results_df['outcome'] == 'PASS').sum()
        fails   = (results_df['outcome'] == 'FAIL').sum()
        errors  = (results_df['outcome'] == 'ERROR').sum()

        precision = tp / (tp + fp) if (tp + fp) else 0
        recall    = tp / (tp + fn) if (tp + fn) else 0

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0
        )

        # ------------------------------------------
        # Summary
        # ------------------------------------------

        print(f'\nAccuracy : {correct/total*100:.1f}% ({correct}/{total})')
        print(f'Failures : {fails} | Errors: {errors}')

        print(
            f'Precision: {precision:.3f} | '
            f'Recall: {recall:.3f} | '
            f'F1: {f1:.3f}'
        )

        print(f'TP={tp} FP={fp} TN={tn} FN={fn}')

        # ------------------------------------------
        # Confusion Matrix
        # ------------------------------------------

        valid_df = results_df[results_df['got'] != 'Error']

        cm = pd.crosstab(
            valid_df['expected'],
            valid_df['got'],
            rownames=['Expected'],
            colnames=['Predicted']
        )

        print("\nConfusion Matrix:")
        display(cm)

        # ------------------------------------------
        # Classification Report
        # ------------------------------------------

        print("\nClassification Report:")

        report_df = pd.DataFrame(
            classification_report(
                valid_df['expected'],
                valid_df['got'],
                output_dict=True,
                zero_division=0
            )
        ).transpose()

        display(report_df)

        # ------------------------------------------
        # Show Failed Cases
        # ------------------------------------------

        failed_cases = results_df[
            results_df['outcome'] != 'PASS'
        ][['id', 'expected', 'got', 'score', 'detail']]

        if len(failed_cases) > 0:
            print("\nFailed Cases:")
            display(failed_cases)


# --------------------------------------------------
# Run
# --------------------------------------------------

thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

threshold_sweep(
    df=df,
    thresholds=thresholds,
    tool_fn=tool_fn,
    PRINCIPLE=PRINCIPLE
)


PRINCIPLE: SRP | THRESHOLD = 0.40


<unknown>:64: SyntaxWarning: invalid escape sequence '\/'
<unknown>:61: SyntaxWarning: invalid escape sequence '\/'



Accuracy : 68.8% (441/641)
Failures : 200 | Errors: 0
Precision: 0.000 | Recall: 0.000 | F1: 0.000
TP=0 FP=0 TN=441 FN=200

Confusion Matrix:


Predicted,Pass
Expected,
Pass,441
Violation,200



Classification Report:


,precision,recall,f1-score,support
Pass,0.687988,1.000000,0.815157,441.000000
Violation,0.000000,0.000000,0.000000,200.000000
accuracy,0.687988,0.687988,0.687988,0.687988
macro avg,0.343994,0.500000,0.407579,641.000000
weighted avg,0.473327,0.687988,0.560818,641.000000



Failed Cases:


,id,expected,got,score,detail
0,1,Violation,Pass,0.0,"[{'class': 'Writer', 'status': 'Violation', 'r..."
6,7,Violation,Pass,0.0,"[{'class': 'User', 'status': 'Violation', 'rea..."
11,12,Violation,Pass,0.0,"[{'class': 'Car', 'status': 'Pass', 'reason': ..."
12,13,Violation,Pass,0.0,"[{'class': 'Order', 'status': 'Violation', 're..."
17,18,Violation,Pass,0.0,"[{'class': 'Order', 'status': 'Violation', 're..."
...,...,...,...,...,...
626,627,Violation,Pass,0.0,"[{'class': 'Order', 'status': 'Violation', 're..."
629,630,Violation,Pass,0.0,"[{'class': 'Order', 'status': 'Pass', 'reason'..."
631,632,Violation,Pass,0.0,"[{'class': 'PowerOutlet', 'status': 'Pass', 'r..."
634,635,Violation,Pass,0.0,"[{'class': 'Car', 'status': 'Pass', 'reason': ..."


## 6 · Error summary

In [9]:
error_df = results_df[results_df['outcome'] == 'ERROR'].reset_index(drop=True)
print(f'Total errors: {len(error_df)}\n')

print('── Exception types ──────────────────────────────────')
display(error_df['exc_type'].value_counts().to_frame('count'))

print('\n── Most common error messages ───────────────────────')
display(error_df['exc_msg'].str[:120].value_counts().head(15).to_frame('count'))

Total errors: 7

── Exception types ──────────────────────────────────


,count
exc_type,
,7



── Most common error messages ───────────────────────


,count
exc_msg,
,7


## 7 · Full traceback for a specific error row

In [10]:
INSPECT_ID = '0'   # ← change to any id from the error table above

row = error_df[error_df['id'] == INSPECT_ID]
if row.empty:
    print(f'No error row with id={INSPECT_ID!r}')
else:
    r = row.iloc[0]
    print(f'ID        : {r["id"]}')
    print(f'Exception : {r["exc_type"]}: {r["exc_msg"]}')
    print(f'Expected  : {r["expected"]}')
    print('\n── Code ─────────────────────────────────────────────')
    print(r['code'])
    print('\n── Full traceback ───────────────────────────────────')
    print(r['exc_tb'])

No error row with id='0'


## 8 · Reproduce a single row live (reloads the tool — no kernel restart needed)

In [11]:
REPRO_ID = '0'   # ← change to the id you want to reproduce

importlib.reload(mod)
tool_fn = getattr(mod, fn_name)

row = results_df[results_df['id'] == REPRO_ID]
if row.empty:
    print(f'No row with id={REPRO_ID!r}')
else:
    code = row.iloc[0]['code']
    print('── Code ─────────────────────────────────────────────')
    print(code)
    print('\n── Live tool output ─────────────────────────────────')
    try:
        print(tool_fn(code))
    except Exception:
        traceback.print_exc()

No row with id='0'


## 9 · Save cleaned dataset to Excel

In [12]:
CLEAN_PATH = r'python-codes-clean.xlsx'

save_df = df.drop(columns=['code_raw'])
save_df.to_excel(CLEAN_PATH, index=False)
print(f'✅ Cleaned dataset saved → {CLEAN_PATH}  ({len(save_df)} rows)')

✅ Cleaned dataset saved → python-codes-clean.xlsx  (641 rows)
